# Fiche 9 — Conditions et classements : CASE WHEN, RANK, moyennes

| | |
|---|---|
| 🎯 **Objectif** | Compter selon une condition, classer à l'intérieur de groupes, et choisir quelle moyenne on calcule |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 8 |
| 🔗 **Dans le projet** | Partie 3 (comparaisons, top N par groupe, ratios) |

> **L'idée clé.** Trois outils, chacun à son tour : `CASE WHEN` pour trier les lignes en catégories, `RANK` pour classer dans chaque groupe, et la question « quelle moyenne ? » avant tout calcul de ratio.

In [1]:
import sqlite3

import pandas as pd

# Ouvre TA base si elle est prête ; sinon, la base de démonstration du dossier (demande-la à ton mentor
# si elle ne t'a pas été transmise). Ouverture en lecture seule : les exemples ne peuvent rien abîmer.
base = sqlite3.connect("file:../data/processed/dataimmo.sqlite?mode=ro", uri=True)


def sql(requete: str) -> pd.DataFrame:
    return pd.read_sql(requete, base)

## 1. CASE WHEN : une condition dans le SELECT

`CASE WHEN condition THEN valeur ELSE autre END` attribue une valeur selon une condition. Combiné à `SUM`, il compte
les lignes qui vérifient la condition. Part des ventes à plusieurs lots, pour chaque type de bien :

In [2]:
sql("""
SELECT
    b.type_local,
    SUM(CASE WHEN v.nombre_lots > 1 THEN 1 ELSE 0 END) AS plusieurs_lots,
    ROUND(SUM(CASE WHEN v.nombre_lots > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS part_pct
FROM vente AS v
INNER JOIN bien AS b ON b.id_bien = v.id_bien
GROUP BY b.type_local
""")

,type_local,plusieurs_lots,part_pct
0,Appartement,11218,35.8
1,Maison,278,10.0


## 2. Top N **par groupe** : RANK() OVER (PARTITION BY …)

« Les 2 types de voie les plus fréquents **de chaque** département » : `ORDER BY … LIMIT 2` donnerait le top global.
`RANK() OVER (PARTITION BY code_departement ORDER BY …)` numérote **à l'intérieur** de chaque département :

In [3]:
sql("""
WITH voies_par_departement AS (
    SELECT c.code_departement, b.type_voie, COUNT(*) AS nb_biens
    FROM bien AS b
    INNER JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
    WHERE b.type_voie IS NOT NULL
    GROUP BY c.code_departement, b.type_voie
),
classement AS (
    SELECT *, RANK() OVER (PARTITION BY code_departement ORDER BY nb_biens DESC) AS rang
    FROM voies_par_departement
)
SELECT code_departement, rang, type_voie, nb_biens
FROM classement
WHERE rang <= 2
LIMIT 8
""")

,code_departement,rang,type_voie,nb_biens
0,01,1,RUE,111
1,01,2,CHE,18
2,02,1,RUE,58
3,02,2,PL,9
4,03,1,RUE,13
5,03,2,AV,1
6,03,2,IMP,1
7,03,2,RES,1


Regarde le département 03 : trois types de voie sont à égalité (1 bien chacun) et reçoivent tous le rang 2.
`RANK` donne le **même rang aux ex æquo** : un « top 2 » peut donc renvoyer plus de 2 lignes. Si l'on veut
exactement 2 lignes, `ROW_NUMBER()` départage arbitrairement ; il faut alors le dire.

Sans fonction de fenêtre, une solution plus simple pour débuter : une requête `ORDER BY … LIMIT 2` par département,
assemblées avec `UNION ALL` :

In [4]:
sql("""
SELECT * FROM (SELECT '33' AS dep, type_voie, COUNT(*) AS nb FROM bien
               WHERE id_codedep_codecommune LIKE '33%' AND type_voie IS NOT NULL
               GROUP BY type_voie ORDER BY nb DESC LIMIT 2)
UNION ALL
SELECT * FROM (SELECT '35' AS dep, type_voie, COUNT(*) AS nb FROM bien
               WHERE id_codedep_codecommune LIKE '35%' AND type_voie IS NOT NULL
               GROUP BY type_voie ORDER BY nb DESC LIMIT 2)
""")

,dep,type_voie,nb
0,33,RUE,610
1,33,AV,196
2,35,RUE,261
3,35,AV,41


⚠️ Le classement se fait sur la **valeur exacte** : arrondir avec `ROUND` avant de classer peut créer de fausses
égalités. On arrondit seulement dans le `SELECT` final.

## 3. ⚠️ Piège : « la moyenne d'un ratio » est ambiguë

Rapport entre surface Carrez et surface réelle bâtie : deux calculs, deux résultats.

In [5]:
sql("""
SELECT
    ROUND(AVG(surface_carrez * 1.0 / surface_reelle_bati), 3)  AS moyenne_des_ratios,
    ROUND(SUM(surface_carrez) / SUM(surface_reelle_bati), 3)   AS ratio_des_totaux,
    ROUND(MAX(surface_carrez * 1.0 / surface_reelle_bati), 1)  AS ratio_maximum,
    COUNT(*)                                                    AS nb_biens
FROM bien
WHERE surface_reelle_bati > 0
""")

,moyenne_des_ratios,ratio_des_totaux,ratio_maximum,nb_biens
0,1.031,1.016,97.2,34169


- La **moyenne des ratios** donne le même poids à chaque bien ; le **ratio des totaux** donne plus de poids aux
  grandes surfaces. Aucun n'est faux, mais il faut **écrire lequel on calcule**.
- Le maximum (un bien dont la surface Carrez vaut près de 100 fois la surface bâtie !) est une erreur de saisie.
  Quelques valeurs de ce genre suffisent à tirer une moyenne : on affiche toujours le **nombre de lignes** et on
  regarde les extrêmes avant de conclure.

### ✍️ À toi de jouer

Écart en % entre la **valeur foncière moyenne** des ventes à 1 lot et celle des ventes à plusieurs lots, vu depuis les ventes à 1 lot. Utilise `AVG(CASE WHEN …)` dans une CTE.

In [6]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
WITH moyennes AS (
    SELECT
        AVG(CASE WHEN nombre_lots = 1 THEN valeur_fonciere END) AS un_lot,
        AVG(CASE WHEN nombre_lots > 1 THEN valeur_fonciere END) AS plusieurs_lots
    FROM vente
)
SELECT ROUND(un_lot, 0) AS moyenne_1_lot,
       ROUND(plusieurs_lots, 0) AS moyenne_plusieurs_lots,
       ROUND((plusieurs_lots - un_lot) * 100.0 / un_lot, 1) AS ecart_pct
FROM moyennes;
```

`AVG(CASE WHEN … THEN valeur END)` ignore les lignes où la condition est fausse (le `CASE` y vaut `NULL`) : deux moyennes en une seule lecture de la table. « Vu depuis les ventes à 1 lot » fixe le dénominateur.
</details>

## ✅ À retenir

1. `SUM(CASE WHEN … THEN 1 ELSE 0 END)` compte les lignes qui vérifient une condition.
2. Top N par groupe : `RANK() OVER (PARTITION BY … ORDER BY …)`, puis filtre sur le rang (ou `UNION ALL`).
3. Classer sur la valeur exacte, arrondir à l'affichage.
4. Moyenne des ratios ou ratio des totaux : il faut l'écrire ; et toujours regarder les extrêmes et l'effectif.